In [ ]:
# ============================================================
# 0. Imports
# ============================================================
import random
import os
import atexit
import multiprocessing as mp
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import dijkstra
from scipy.stats import spearmanr, pearsonr
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================
NODES_CSV = "output/processed_fishtree_nodes.csv"
EDGE_INDEX_NPY = "output/processed_fishtree_edge_index.npy"
EDGE_WEIGHT_NPY = "output/processed_fishtree_edge_weight.npy"

OUTPUT_DIR = "output/hyperbolic_graph_embedding"
DIST_MATRIX_NAME = "tree_distance_matrix.npy"
CHECKPOINT_NAME = "node_embedding_model_hyperbolic.pt"

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Model / optimization
EMB_DIM = 256                 # hyperbolic often needs fewer dims than Euclidean
LR = 1e-5
WEIGHT_DECAY = 0.0
NUM_STEPS = 300_000
BATCH_SIZE = 4096 // 2

# Sampler parallelism
# Set to 1 for serial sampling. Set to None to use all CPU cores.
# A small value like 4-8 is usually a good starting point because each worker
# returns a chunk of triplets and the training loop consumes one batch at a time.
SAMPLER_NUM_WORKERS = min(8, os.cpu_count() or 1)
SAMPLER_CHUNK_SIZE = 512 // 2
SAMPLER_MP_START_METHOD = "fork"  # fastest on Linux; falls back safely if unavailable

# Hyperbolic geometry
BALL_EPS = 1e-5
INIT_SCALE = 1e-3

# Loss
LAMBDA_PAIR = 0.5
TRIPLET_MARGIN = 0.0
USE_WEIGHTED_DISTANCE_LOSS = True

# Scale tree distance before comparing to hyperbolic distance
# Start with this; tune later.
TREE_DISTANCE_SCALE = 1.0

# Distance binning / sampler
NUM_DISTANCE_BINS = 100
CLIP_DIST_PERCENTILE = 95.0

NEG_OFFSET_ALPHA = 1.5
NEG_OFFSET_UNIFORM_MIX = 0.10

INCLUDE_OVERFLOW_AS_NEGATIVE = True

COMBINED_PROB_TARGET_SLOPE = 0.2

# Dense matrix construction
DIJKSTRA_BLOCK_SIZE = 256

# Evaluation
EVAL_NUM_PAIR_SAMPLES = 3000
EVAL_NUM_ANCHORS = 200
EVAL_CANDIDATE_SIZE = 2000

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Using device:", DEVICE)

In [ ]:
# ============================================================
# 2. Load tree files
# ============================================================
nodes_df = pd.read_csv(NODES_CSV)
edge_index = np.load(EDGE_INDEX_NPY)
edge_weight = np.load(EDGE_WEIGHT_NPY)

num_nodes = len(nodes_df)

is_leaf = nodes_df["is_leaf"].astype(bool).to_numpy()
leaf_ids = np.where(is_leaf)[0]
internal_ids = np.where(~is_leaf)[0]

print("num_nodes:", num_nodes)
print("num leaves:", len(leaf_ids))
print("num internal:", len(internal_ids))

In [ ]:
# ============================================================
# 3. Build / load dense tree-distance matrix
# ============================================================
def build_sparse_graph(edge_index, edge_weight, num_nodes):
    src = edge_index[0].astype(np.int64)
    dst = edge_index[1].astype(np.int64)
    data = edge_weight.astype(np.float64)
    return csr_matrix((data, (src, dst)), shape=(num_nodes, num_nodes))


def build_distance_matrix_blockwise(graph, num_nodes, out_path, block_size=256):
    out_path = Path(out_path)

    dist_mem = np.lib.format.open_memmap(
        out_path,
        mode="w+",
        dtype=np.float32,
        shape=(num_nodes, num_nodes),
    )

    for start in tqdm(range(0, num_nodes, block_size), desc="Computing distance matrix"):
        end = min(start + block_size, num_nodes)
        indices = np.arange(start, end, dtype=np.int64)

        block = dijkstra(
            csgraph=graph,
            directed=False,
            indices=indices,
            return_predecessors=False,
        )

        dist_mem[start:end, :] = block.astype(np.float32)

    dist_mem.flush()
    del dist_mem


output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
dist_matrix_path = output_dir / DIST_MATRIX_NAME

if not dist_matrix_path.exists():
    graph = build_sparse_graph(edge_index, edge_weight, num_nodes)
    build_distance_matrix_blockwise(
        graph=graph,
        num_nodes=num_nodes,
        out_path=dist_matrix_path,
        block_size=DIJKSTRA_BLOCK_SIZE,
    )

D = np.load(dist_matrix_path)
print("D:", D.shape, D.dtype)
print("D memory GB:", D.nbytes / (1024 ** 3))

In [ ]:
# ============================================================
# 4. Estimate tau / clip distance
# ============================================================
def sample_distance_distribution_from_dense(D, num_samples=100_000):
    n = D.shape[0]
    dists = []

    while len(dists) < num_samples:
        u = random.randrange(n)
        v = random.randrange(n)
        if u != v:
            dists.append(float(D[u, v]))

    return np.array(dists, dtype=np.float32)


dist_samples = sample_distance_distribution_from_dense(D)
median_distance = float(np.median(dist_samples))
clip_distance = float(np.percentile(dist_samples, CLIP_DIST_PERCENTILE))

print("median distance:", median_distance)
print("clip_distance:", clip_distance)

for q in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(f"p{q:2d} = {np.percentile(dist_samples, q):.4f}")

In [ ]:
# ============================================================
# 5. Compact equal-width band index
# ============================================================
def build_compact_band_index_from_dense(D, clip_distance, num_regular_bins):
    n = D.shape[0]
    num_total_bins = num_regular_bins + 1

    sorted_ids = np.empty((n, n - 1), dtype=np.int32)
    bin_offsets = np.empty((n, num_total_bins + 1), dtype=np.int32)
    bin_edges = np.linspace(0.0, clip_distance, num_regular_bins + 1, dtype=np.float32)

    for u in tqdm(range(n), desc="Building compact band index"):
        row = D[u]

        order = np.argsort(row, kind="stable")
        order = order[order != u]

        sorted_ids[u, :] = order.astype(np.int32, copy=False)

        sorted_dists = row[order]
        ends = np.searchsorted(sorted_dists, bin_edges[1:], side="right").astype(np.int32)

        bin_offsets[u, 0] = 0
        bin_offsets[u, 1:num_regular_bins + 1] = ends
        bin_offsets[u, num_total_bins] = n - 1

    return {
        "sorted_ids": sorted_ids,
        "bin_offsets": bin_offsets,
        "bin_edges": bin_edges,
        "clip_distance": float(clip_distance),
        "num_regular_bins": int(num_regular_bins),
        "num_total_bins": int(num_total_bins),
    }


band_index = build_compact_band_index_from_dense(
    D=D,
    clip_distance=clip_distance,
    num_regular_bins=NUM_DISTANCE_BINS,
)

sorted_ids = band_index["sorted_ids"]
bin_offsets = band_index["bin_offsets"]

print("sorted_ids:", sorted_ids.shape, sorted_ids.dtype)
print("bin_offsets:", bin_offsets.shape, bin_offsets.dtype)

In [ ]:
# ============================================================
# Bin occupancy histogram (all anchors)
# ============================================================
n = D.shape[0]
num_bins = bin_offsets.shape[1] - 1

# Bin sizes for every anchor and every bin
bin_counts = bin_offsets[:, 1:] - bin_offsets[:, :-1]

# Statistics over all anchors
mean_counts = bin_counts.mean(axis=0)
std_counts = bin_counts.std(axis=0)
empty_fraction_per_bin = (bin_counts == 0).mean(axis=0)

# ============================================================
# Plot: average number of nodes per bin
# ============================================================
plt.figure(figsize=(10, 5))
plt.plot(mean_counts, label="mean nodes per bin")
plt.fill_between(
    np.arange(num_bins),
    mean_counts - std_counts,
    mean_counts + std_counts,
    alpha=0.3,
    label="±1 std",
)

plt.xlabel("Bin index")
plt.ylabel("Number of nodes")
plt.title("Bin occupancy over all anchors")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# ============================================================
# Plot: fraction of anchors with empty bin
# ============================================================
plt.figure(figsize=(10, 4))
plt.plot(empty_fraction_per_bin)
plt.xlabel("Bin index")
plt.ylabel("Fraction empty")
plt.title("Fraction of anchors with empty bin")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.show()


print("bin_counts:", bin_counts.shape, bin_counts.dtype)
print("Overall fraction of empty anchor-bin pairs:", np.mean(bin_counts == 0))
print("Mean bin size first 10 bins:", mean_counts[:10])
print("Mean bin size last 10 bins:", mean_counts[-10:])

In [ ]:
# ============================================================
# 6. Unified sampler
# ============================================================
def make_power_law_probs(num_items, alpha=1.0):
    x = np.arange(num_items, dtype=np.float64)
    probs = 1.0 / ((1.0 + x) ** alpha)
    probs /= probs.sum()
    return probs


def make_mixture_power_law_probs(num_items, local_alpha=1.5, uniform_mix=0.1):
    local = make_power_law_probs(num_items, alpha=local_alpha)
    uniform = np.ones(num_items, dtype=np.float64) / num_items
    probs = (1.0 - uniform_mix) * local + uniform_mix * uniform
    probs /= probs.sum()
    return probs

def make_combined_uniform_positive_probs(
    empty_fraction_regular,
    neg_offset_probs,
    num_iters=2000,
    lr=0.2,
    min_nonempty_fraction=1e-6,
    target_slope=0.0
):
    nonempty_fraction = 1.0 - empty_fraction_regular
    nonempty_fraction = np.maximum(nonempty_fraction, min_nonempty_fraction)

    n = len(nonempty_fraction)

    # Start from effective-positive uniform
    effective_pos = np.ones(n, dtype=np.float64) / n

    #target = np.ones(n, dtype=np.float64) / n
    target = np.linspace(1.0, 1.0-target_slope, num=n)
    target /= target.sum()

    for _ in range(num_iters):
        neg_bin = np.convolve(effective_pos, neg_offset_probs)[:n]
        combined = effective_pos + neg_bin - effective_pos * neg_bin
        combined /= combined.sum()

        # If combined is too high in a bin, reduce effective positive there.
        # If combined is too low, increase it.
        ratio = target / np.maximum(combined, 1e-12)
        effective_pos *= ratio ** lr
        effective_pos /= effective_pos.sum()

    # Convert desired effective positive distribution back into sampling probs
    # before empty-bin rejection:
    sampling_pos = effective_pos / nonempty_fraction
    sampling_pos /= sampling_pos.sum()

    # Recompute actual effective positive after availability correction
    effective_pos_actual = sampling_pos * nonempty_fraction
    effective_pos_actual /= effective_pos_actual.sum()

    neg_bin = np.convolve(effective_pos_actual, neg_offset_probs)[:n]
    combined = effective_pos_actual + neg_bin - effective_pos_actual * neg_bin
    combined /= combined.sum()

    return sampling_pos, effective_pos_actual, neg_bin, combined

empty_fraction_regular = empty_fraction_per_bin[:NUM_DISTANCE_BINS]

neg_offset_probs = make_mixture_power_law_probs(
    NUM_DISTANCE_BINS,
    local_alpha=NEG_OFFSET_ALPHA,
    uniform_mix=NEG_OFFSET_UNIFORM_MIX,
)

pos_bin_probs, effective_probs, neg_bin_probs, combined_probs = (
    make_combined_uniform_positive_probs(
        empty_fraction_regular=empty_fraction_regular,
        neg_offset_probs=neg_offset_probs,
        num_iters=2000,
        lr=0.2,
        target_slope=COMBINED_PROB_TARGET_SLOPE,
    )
)

print("First 5 bin offset probs:", neg_offset_probs[:5])

plt.figure(figsize=(9, 5))
plt.plot(pos_bin_probs, label="sampling P(positive bin)")
plt.plot(effective_probs, label="effective P(positive and non-empty)")
plt.plot(neg_bin_probs, label="approx P(negative bin)")
plt.plot(combined_probs, label="approx combined P(pos or neg)", linewidth=2)
plt.xlabel("Bin index")
plt.ylabel("Probability")
plt.title("Sampling distributions")
plt.legend()
plt.grid(alpha=0.3)
plt.show()
    
    

In [ ]:
# ============================================================
# 7. Unified sampler with optional multi-core batch sampling
# ============================================================
_SAMPLER_WORKER_STATE = None


def _init_sampler_worker(
    D,
    sorted_ids,
    bin_offsets,
    num_regular_bins,
    include_overflow_as_negative,
    pos_bin_probs,
    neg_offset_probs,
    max_anchor_tries,
    max_bin_tries,
):
    """Initializer for multiprocessing workers.

    With the default Linux "fork" start method, the large arrays are shared
    copy-on-write instead of copied into every worker.
    """
    global _SAMPLER_WORKER_STATE
    _SAMPLER_WORKER_STATE = {
        "D": D,
        "sorted_ids": sorted_ids,
        "bin_offsets": bin_offsets,
        "num_nodes": D.shape[0],
        "num_regular_bins": int(num_regular_bins),
        "overflow_bin": int(num_regular_bins),
        "include_overflow_as_negative": bool(include_overflow_as_negative),
        "pos_bin_probs": np.asarray(pos_bin_probs, dtype=np.float64),
        "neg_offset_probs": np.asarray(neg_offset_probs, dtype=np.float64),
        "max_anchor_tries": int(max_anchor_tries),
        "max_bin_tries": int(max_bin_tries),
    }


def _sample_triplet_from_state(state, rng):
    D = state["D"]
    sorted_ids = state["sorted_ids"]
    bin_offsets = state["bin_offsets"]
    num_nodes = state["num_nodes"]
    num_regular_bins = state["num_regular_bins"]
    overflow_bin = state["overflow_bin"]
    include_overflow = state["include_overflow_as_negative"]
    pos_bin_probs = state["pos_bin_probs"]
    neg_offset_probs = state["neg_offset_probs"]
    max_anchor_tries = state["max_anchor_tries"]
    max_bin_tries = state["max_bin_tries"]

    for _ in range(max_anchor_tries):
        anchor = int(rng.integers(num_nodes))

        b_pos = None
        for _ in range(max_bin_tries):
            b = int(rng.choice(num_regular_bins, p=pos_bin_probs))
            if int(bin_offsets[anchor, b + 1]) > int(bin_offsets[anchor, b]):
                b_pos = b
                break
        if b_pos is None:
            continue

        b_neg = None
        max_regular_offset = num_regular_bins - 1 - b_pos
        for _ in range(max_bin_tries):
            offset = int(rng.choice(len(neg_offset_probs), p=neg_offset_probs))
            if offset <= max_regular_offset:
                candidate = b_pos + offset
            else:
                candidate = overflow_bin if include_overflow else num_regular_bins - 1

            if int(bin_offsets[anchor, candidate + 1]) > int(bin_offsets[anchor, candidate]):
                b_neg = candidate
                break
        if b_neg is None:
            continue

        start_pos = int(bin_offsets[anchor, b_pos])
        end_pos = int(bin_offsets[anchor, b_pos + 1])
        start_neg = int(bin_offsets[anchor, b_neg])
        end_neg = int(bin_offsets[anchor, b_neg + 1])

        if b_neg == b_pos:
            if end_pos - start_pos < 2:
                continue
            i = int(rng.integers(start_pos, end_pos - 1))
            j = int(rng.integers(i + 1, end_pos))
            v = int(sorted_ids[anchor, i])
            w = int(sorted_ids[anchor, j])
            d_pos = float(D[anchor, v])
            d_neg = float(D[anchor, w])
            if d_pos > d_neg:
                v, w = w, v
                d_pos, d_neg = d_neg, d_pos
            elif d_pos == d_neg:
                continue
        else:
            i = int(rng.integers(start_pos, end_pos))
            j = int(rng.integers(start_neg, end_neg))
            v = int(sorted_ids[anchor, i])
            w = int(sorted_ids[anchor, j])
            d_pos = float(D[anchor, v])
            d_neg = float(D[anchor, w])
            if d_pos > d_neg:
                v, w = w, v
                d_pos, d_neg = d_neg, d_pos
            if d_pos == d_neg:
                continue

        return {
            "anchor": anchor,
            "pos": v,
            "neg": w,
            "d_pos": d_pos,
            "d_neg": d_neg,
            "b_pos": b_pos,
            "b_neg": b_neg,
            "bin_offset": b_neg - b_pos if b_neg < overflow_bin else overflow_bin - b_pos,
        }

    return None


def _sample_triplets_worker(args):
    num_triplets, seed = args
    rng = np.random.default_rng(seed)
    out = []
    state = _SAMPLER_WORKER_STATE
    while len(out) < num_triplets:
        t = _sample_triplet_from_state(state, rng)
        if t is not None:
            out.append(t)
    return out


class UnifiedDistanceBinSampler:
    def __init__(
        self,
        D,
        sorted_ids,
        bin_offsets,
        num_regular_bins,
        include_overflow_as_negative=True,
        pos_bin_probs=None,
        neg_offset_probs=None,
        max_anchor_tries=100,
        max_bin_tries=100,
        num_workers=1,
        chunk_size=512,
        mp_start_method="fork",
        seed=42,
    ):
        self.D = D
        self.sorted_ids = sorted_ids
        self.bin_offsets = bin_offsets

        self.num_nodes = D.shape[0]
        self.num_regular_bins = num_regular_bins
        self.overflow_bin = num_regular_bins
        self.num_total_bins = num_regular_bins + 1

        self.include_overflow_as_negative = include_overflow_as_negative
        self.max_anchor_tries = max_anchor_tries
        self.max_bin_tries = max_bin_tries

        self.pos_bin_probs = np.asarray(pos_bin_probs, dtype=np.float64)
        self.pos_bin_probs /= self.pos_bin_probs.sum()

        self.neg_offset_probs = np.asarray(neg_offset_probs, dtype=np.float64)
        self.neg_offset_probs /= self.neg_offset_probs.sum()

        if num_workers is None:
            num_workers = os.cpu_count() or 1
        self.num_workers = max(1, int(num_workers))
        self.chunk_size = max(1, int(chunk_size))
        self.mp_start_method = mp_start_method
        self._rng = np.random.default_rng(seed)
        self._pool = None

        self._state = {
            "D": self.D,
            "sorted_ids": self.sorted_ids,
            "bin_offsets": self.bin_offsets,
            "num_nodes": self.num_nodes,
            "num_regular_bins": self.num_regular_bins,
            "overflow_bin": self.overflow_bin,
            "include_overflow_as_negative": self.include_overflow_as_negative,
            "pos_bin_probs": self.pos_bin_probs,
            "neg_offset_probs": self.neg_offset_probs,
            "max_anchor_tries": self.max_anchor_tries,
            "max_bin_tries": self.max_bin_tries,
        }

        atexit.register(self.close)

    def close(self):
        if self._pool is not None:
            self._pool.close()
            self._pool.join()
            self._pool = None

    def _get_pool(self):
        if self._pool is None:
            try:
                ctx = mp.get_context(self.mp_start_method)
            except ValueError:
                ctx = mp.get_context()

            self._pool = ctx.Pool(
                processes=self.num_workers,
                initializer=_init_sampler_worker,
                initargs=(
                    self.D,
                    self.sorted_ids,
                    self.bin_offsets,
                    self.num_regular_bins,
                    self.include_overflow_as_negative,
                    self.pos_bin_probs,
                    self.neg_offset_probs,
                    self.max_anchor_tries,
                    self.max_bin_tries,
                ),
            )
        return self._pool

    def sample_triplet(self):
        return _sample_triplet_from_state(self._state, self._rng)

    def sample_batch(self, batch_size):
        if self.num_workers <= 1:
            batch = []
            while len(batch) < batch_size:
                t = self.sample_triplet()
                if t is not None:
                    batch.append(t)
            return batch

        # Split the requested batch into moderately large chunks.  This keeps
        # inter-process communication low while still spreading work across cores.
        chunks = []
        remaining = int(batch_size)
        while remaining > 0:
            n = min(self.chunk_size, remaining)
            seed = int(self._rng.integers(0, np.iinfo(np.uint32).max))
            chunks.append((n, seed))
            remaining -= n

        pool = self._get_pool()
        batch = []
        for part in pool.imap_unordered(_sample_triplets_worker, chunks):
            batch.extend(part)
        return batch[:batch_size]


sampler = UnifiedDistanceBinSampler(
    D=D,
    sorted_ids=sorted_ids,
    bin_offsets=bin_offsets,
    num_regular_bins=NUM_DISTANCE_BINS,
    include_overflow_as_negative=INCLUDE_OVERFLOW_AS_NEGATIVE,
    pos_bin_probs=pos_bin_probs,
    neg_offset_probs=neg_offset_probs,
    num_workers=SAMPLER_NUM_WORKERS,
    chunk_size=SAMPLER_CHUNK_SIZE,
    mp_start_method=SAMPLER_MP_START_METHOD,
    seed=SEED,
)

print(f"Sampler workers: {sampler.num_workers}")


In [ ]:
# ============================================================
# Empirical bin distribution from sampler
# ============================================================

NUM_SAMPLES = 200_000  # increase for smoother curves

num_bins = sampler.num_regular_bins  # excludes overflow
overflow_bin = sampler.overflow_bin

pos_counts = np.zeros(num_bins, dtype=np.int64)
neg_counts = np.zeros(num_bins, dtype=np.int64)
combined_counts = np.zeros(num_bins, dtype=np.int64)

overflow_pos = 0
overflow_neg = 0

for _ in tqdm(range(NUM_SAMPLES), desc="Sampling triplets"):
    t = sampler.sample_triplet()
    if t is None:
        continue

    b_pos = t["b_pos"]
    b_neg = t["b_neg"]

    # Positive bin
    if b_pos < num_bins:
        pos_counts[b_pos] += 1
        combined_counts[b_pos] += 1
    else:
        overflow_pos += 1

    # Negative bin
    if b_neg < num_bins:
        neg_counts[b_neg] += 1
        combined_counts[b_neg] += 1
    else:
        overflow_neg += 1

# Normalize
pos_probs = pos_counts / pos_counts.sum()
neg_probs = neg_counts / neg_counts.sum()
combined_probs = combined_counts / combined_counts.sum()

# ============================================================
# Plot
# ============================================================

plt.figure(figsize=(10, 5))
plt.plot(pos_probs, label="Empirical P(positive bin)")
plt.plot(neg_probs, label="Empirical P(negative bin)")
plt.plot(combined_probs, label="Empirical combined P(pos ∪ neg)", linewidth=2)

plt.axhline(1 / num_bins, linestyle="--", label="uniform target")

plt.xlabel("Bin index")
plt.ylabel("Probability")
plt.title(f"Empirical bin distributions ({NUM_SAMPLES} samples)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# ============================================================
# 7. Hyperbolic geometry helpers
# ============================================================
def project_to_poincare_ball(x, eps=1e-5):
    """
    Projects points to the open unit ball.
    """
    norm = torch.norm(x, dim=-1, keepdim=True).clamp_min(eps)
    max_norm = 1.0 - eps
    scale = torch.clamp(max_norm / norm, max=1.0)
    return x * scale


def poincare_distance(x, y, eps=1e-5):
    """
    Pairwise row-wise Poincare distance.
    x, y: [B, D]
    returns: [B]
    """
    x = project_to_poincare_ball(x, eps)
    y = project_to_poincare_ball(y, eps)

    x2 = torch.sum(x * x, dim=-1)
    y2 = torch.sum(y * y, dim=-1)
    diff2 = torch.sum((x - y) ** 2, dim=-1)

    denom = (1.0 - x2) * (1.0 - y2)
    z = 1.0 + 2.0 * diff2 / torch.clamp(denom, min=eps)

    return torch.acosh(torch.clamp(z, min=1.0 + eps))


def poincare_pairwise_distance(x, y, eps=1e-5):
    """
    Full pairwise Poincare distance.
    x: [B, D]
    y: [N, D]
    returns: [B, N]
    """
    x = project_to_poincare_ball(x, eps)
    y = project_to_poincare_ball(y, eps)

    x2 = torch.sum(x * x, dim=-1, keepdim=True)      # [B, 1]
    y2 = torch.sum(y * y, dim=-1).unsqueeze(0)       # [1, N]
    diff2 = torch.cdist(x, y, p=2) ** 2              # [B, N]

    denom = (1.0 - x2) * (1.0 - y2)
    z = 1.0 + 2.0 * diff2 / torch.clamp(denom, min=eps)

    return torch.acosh(torch.clamp(z, min=1.0 + eps))

In [ ]:
# ============================================================
# 8. Hyperbolic node embedding model
# ============================================================
class HyperbolicNodeEmbeddingModel(nn.Module):
    def __init__(self, num_nodes, emb_dim=64, init_scale=1e-3, eps=1e-5):
        super().__init__()
        self.embedding = nn.Embedding(num_nodes, emb_dim)
        self.eps = eps

        with torch.no_grad():
            self.embedding.weight.normal_(mean=0.0, std=init_scale)
            self.embedding.weight.copy_(project_to_poincare_ball(self.embedding.weight, eps=self.eps))

    def forward(self, node_ids):
        z = self.embedding(node_ids)
        return project_to_poincare_ball(z, eps=self.eps)

    def get_all_embeddings(self):
        return project_to_poincare_ball(self.embedding.weight, eps=self.eps)

    @torch.no_grad()
    def project_parameters(self):
        self.embedding.weight.copy_(project_to_poincare_ball(self.embedding.weight, eps=self.eps))

In [ ]:
# ============================================================
# 9. Hyperbolic loss
# ============================================================
def hyperbolic_distance_weight(d):
        return 1.0 / (1.0 + d)

def compute_hyperbolic_loss(
    model,
    batch_triplets,
    tree_distance_scale=1.0,
    lambda_pair=0.2,
    triplet_margin=0.0,
    device="cpu",
    eps=1e-5,
    use_weighted_distance=False,
):
    u = torch.tensor([t["anchor"] for t in batch_triplets], dtype=torch.long, device=device)
    v = torch.tensor([t["pos"] for t in batch_triplets], dtype=torch.long, device=device)
    w = torch.tensor([t["neg"] for t in batch_triplets], dtype=torch.long, device=device)

    d_tree_uv = torch.tensor([t["d_pos"] for t in batch_triplets], dtype=torch.float32, device=device)
    d_tree_uw = torch.tensor([t["d_neg"] for t in batch_triplets], dtype=torch.float32, device=device)

    z_u = model(u)
    z_v = model(v)
    z_w = model(w)

    d_hyp_uv = poincare_distance(z_u, z_v, eps=eps)
    d_hyp_uw = poincare_distance(z_u, z_w, eps=eps)

    # Distance triplet loss: positive should be closer than negative.
    triplet_terms = F.relu(d_hyp_uv - d_hyp_uw + triplet_margin)
    triplet_loss = triplet_terms.mean()

    target_uv = tree_distance_scale * d_tree_uv
    target_uw = tree_distance_scale * d_tree_uw

    if use_weighted_distance:
        weight_uv = hyperbolic_distance_weight(d_tree_uv)
        weight_uw = hyperbolic_distance_weight(d_tree_uw)
    else:
        weight_uv = torch.ones_like(d_tree_uv)
        weight_uw = torch.ones_like(d_tree_uw)

    pair_loss = (
        (weight_uv * (d_hyp_uv - target_uv) ** 2).mean()
        + (weight_uw * (d_hyp_uw - target_uw) ** 2).mean()
    )

    total_loss = triplet_loss + lambda_pair * pair_loss

    stats = {
        "loss": float(total_loss.item()),
        "triplet_loss": float(triplet_loss.item()),
        "pair_loss": float(pair_loss.item()),
        "mean_hyp_d_pos": float(d_hyp_uv.mean().item()),
        "mean_hyp_d_neg": float(d_hyp_uw.mean().item()),
        "mean_tree_d_pos": float(d_tree_uv.mean().item()),
        "mean_tree_d_neg": float(d_tree_uw.mean().item()),
        "mean_norm": float(torch.norm(model.get_all_embeddings(), dim=-1).mean().item()),
        "max_norm": float(torch.norm(model.get_all_embeddings(), dim=-1).max().item()),
    }

    return total_loss, stats

In [ ]:
# ============================================================
# 11. Training
# ============================================================
model = HyperbolicNodeEmbeddingModel(
    num_nodes=num_nodes,
    emb_dim=EMB_DIM,
    init_scale=INIT_SCALE,
    eps=BALL_EPS,
).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

history = []

for step in range(NUM_STEPS):
    batch = sampler.sample_batch(BATCH_SIZE)

    optimizer.zero_grad()

    loss, stats = compute_hyperbolic_loss(
        model=model,
        batch_triplets=batch,
        tree_distance_scale=TREE_DISTANCE_SCALE,
        lambda_pair=LAMBDA_PAIR,
        triplet_margin=TRIPLET_MARGIN,
        device=DEVICE,
        eps=BALL_EPS,
        use_weighted_distance=USE_WEIGHTED_DISTANCE_LOSS,
    )

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()
    model.project_parameters()

    history.append(stats)

    if step % 500 == 0:
        print(
            f"step={step:7d} "
            f"loss={stats['loss']:.4f} "
            f"triplet={stats['triplet_loss']:.4f} "
            f"pair={stats['pair_loss']:.4f} "
            f"hyp_pos={stats['mean_hyp_d_pos']:.3f} "
            f"hyp_neg={stats['mean_hyp_d_neg']:.3f} "
            f"mean_norm={stats['mean_norm']:.4f} "
            f"max_norm={stats['max_norm']:.4f}"
        )

In [ ]:
# ============================================================
# 12. Plot training curves
# ============================================================
losses = [x["loss"] for x in history]
triplet_losses = [x["triplet_loss"] for x in history]
pair_losses = [x["pair_loss"] for x in history]
mean_norms = [x["mean_norm"] for x in history]
max_norms = [x["max_norm"] for x in history]

plt.figure(figsize=(8, 4))
plt.plot(losses, label="total")
plt.plot(triplet_losses, label="triplet")
plt.plot(pair_losses, label="pair")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.title("Hyperbolic training losses")
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(mean_norms, label="mean norm")
plt.plot(max_norms, label="max norm")
plt.xlabel("step")
plt.ylabel("Poincaré ball norm")
plt.legend()
plt.title("Embedding norms")
plt.show()

In [ ]:
# ============================================================
# 13. Evaluation helpers
# ============================================================
@torch.no_grad()
def hyperbolic_distance_to_candidates(model, anchor, candidates, device="cpu", eps=1e-5):
    model.eval()

    nodes = torch.tensor([anchor] + list(candidates), dtype=torch.long, device=device)
    z = model(nodes)

    z_anchor = z[0:1]
    z_cands = z[1:]

    dists = poincare_pairwise_distance(z_anchor, z_cands, eps=eps)[0]
    return dists.cpu().numpy()


@torch.no_grad()
def evaluate_triplet_accuracy_hyperbolic(model, sampler, num_triplets=5000, device="cpu", eps=1e-5):
    model.eval()

    records = []

    for _ in range(num_triplets):
        t = sampler.sample_triplet()
        if t is None:
            continue

        u, v, w = t["anchor"], t["pos"], t["neg"]

        z = model(torch.tensor([u, v, w], dtype=torch.long, device=device))
        d_uv = float(poincare_distance(z[0:1], z[1:2], eps=eps).item())
        d_uw = float(poincare_distance(z[0:1], z[2:3], eps=eps).item())

        records.append({
            "correct": float(d_uv < d_uw),
            "hyp_margin": d_uw - d_uv,
            "b_pos": t["b_pos"],
            "b_neg": t["b_neg"],
            "bin_offset": t["bin_offset"],
            "tree_d_pos": t["d_pos"],
            "tree_d_neg": t["d_neg"],
            "tree_d_gap": t["d_neg"] - t["d_pos"],
            "same_bin": int(t["b_pos"] == t["b_neg"]),
        })

    df = pd.DataFrame(records)

    return {
        "overall_accuracy": float(df["correct"].mean()),
        "same_bin_accuracy": float(df.loc[df["same_bin"] == 1, "correct"].mean()),
        "different_bin_accuracy": float(df.loc[df["same_bin"] == 0, "correct"].mean()),
        "mean_hyp_margin": float(df["hyp_margin"].mean()),
        "mean_tree_gap": float(df["tree_d_gap"].mean()),
        "num_triplets": int(len(df)),
    }, df


@torch.no_grad()
def evaluate_recall_at_k_hyperbolic(
    model,
    D,
    node_ids,
    k=10,
    num_anchors=200,
    candidate_size=2000,
    device="cpu",
    eps=1e-5,
):
    model.eval()

    recalls = []
    node_ids = list(node_ids)

    for _ in range(num_anchors):
        anchor = int(random.choice(node_ids))
        candidates = random.sample(node_ids, min(candidate_size, len(node_ids)))

        if anchor in candidates:
            candidates.remove(anchor)

        if len(candidates) < k:
            continue

        tree_dists = D[anchor, candidates]
        true_topk = set([candidates[i] for i in np.argsort(tree_dists)[:k]])

        hyp_dists = hyperbolic_distance_to_candidates(model, anchor, candidates, device=device, eps=eps)
        pred_topk = set([candidates[i] for i in np.argsort(hyp_dists)[:k]])

        recalls.append(len(true_topk & pred_topk) / k)

    return {
        "k": k,
        "num_anchors": len(recalls),
        "mean_recall": float(np.mean(recalls)),
        "std_recall": float(np.std(recalls)),
    }

In [ ]:
# ============================================================
# 14. Run focused evaluation
# ============================================================
triplet_summary, triplet_df = evaluate_triplet_accuracy_hyperbolic(
    model=model,
    sampler=sampler,
    num_triplets=5000,
    device=DEVICE,
    eps=BALL_EPS,
)
print("Triplet:", triplet_summary)

recall_10 = evaluate_recall_at_k_hyperbolic(
    model=model,
    D=D,
    node_ids=leaf_ids,
    k=10,
    num_anchors=EVAL_NUM_ANCHORS,
    candidate_size=EVAL_CANDIDATE_SIZE,
    device=DEVICE,
    eps=BALL_EPS,
)
print("Recall@10:", recall_10)

In [ ]:
# ============================================================
# Plotly: all-bin scatter + per-bin within-anchor correlation
# ============================================================
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import spearmanr, pearsonr

@torch.no_grad()
def plot_all_bins_scatter_with_bin_correlations_hyperbolic_plotly(
    model,
    D,
    sorted_ids,
    bin_offsets,
    node_ids,
    num_anchors=100,
    samples_per_bin_per_anchor=25,
    device="cpu",
    eps=1e-5,
    min_points_per_bin_anchor=10,
    max_bins=None,
    correlation_type="spearman",
    opacity=0.45,
    marker_size=5,
):
    """
    Creates a two-panel Plotly figure:

    Left:
      pooled uncentered tree distance vs hyperbolic distance,
      colored by distance bin.

    Right:
      within-bin correlation calculated separately per anchor/bin,
      then averaged over anchors for each bin with ± std.

    Returns
    -------
    points_df : pd.DataFrame
        All sampled anchor-node points.
    corr_summary_df : pd.DataFrame
        Mean/std correlation per bin.
    corr_per_anchor_df : pd.DataFrame
        One row per anchor/bin.
    fig : plotly Figure
    """
    model.eval()

    node_ids = list(node_ids)
    anchors = random.sample(node_ids, min(num_anchors, len(node_ids)))

    num_bins = bin_offsets.shape[1] - 1
    if max_bins is not None:
        num_bins = min(num_bins, max_bins)

    point_records = []
    corr_records = []

    for anchor in tqdm(anchors, desc="Collecting scatter + bin correlations"):
        anchor = int(anchor)

        for b in range(num_bins):
            start = int(bin_offsets[anchor, b])
            end = int(bin_offsets[anchor, b + 1])
            bin_nodes = sorted_ids[anchor, start:end]

            if len(bin_nodes) < min_points_per_bin_anchor:
                continue

            if len(bin_nodes) > samples_per_bin_per_anchor:
                sampled_nodes = np.random.choice(
                    bin_nodes,
                    size=samples_per_bin_per_anchor,
                    replace=False,
                )
            else:
                sampled_nodes = bin_nodes

            sampled_nodes = sampled_nodes.astype(np.int64)

            tree_dists = D[anchor, sampled_nodes].astype(np.float64)

            hyp_dists = hyperbolic_distance_to_candidates(
                model=model,
                anchor=anchor,
                candidates=sampled_nodes.tolist(),
                device=device,
                eps=eps,
            ).astype(np.float64)

            for node, td, hd in zip(sampled_nodes, tree_dists, hyp_dists):
                point_records.append({
                    "anchor": anchor,
                    "node": int(node),
                    "bin_index": b,
                    "tree_distance": float(td),
                    "hyperbolic_distance": float(hd),
                })

            if len(np.unique(tree_dists)) >= 2 and len(np.unique(hyp_dists)) >= 2:
                if correlation_type == "spearman":
                    corr = spearmanr(tree_dists, hyp_dists).correlation
                elif correlation_type == "pearson":
                    corr = pearsonr(tree_dists, hyp_dists)[0]
                else:
                    raise ValueError("correlation_type must be 'spearman' or 'pearson'")
            else:
                corr = np.nan

            corr_records.append({
                "anchor": anchor,
                "bin_index": b,
                "num_points": len(sampled_nodes),
                "correlation": float(corr) if not np.isnan(corr) else np.nan,
                "tree_range": float(tree_dists.max() - tree_dists.min()),
                "hyp_range": float(hyp_dists.max() - hyp_dists.min()),
            })

    points_df = pd.DataFrame(point_records)
    corr_per_anchor_df = pd.DataFrame(corr_records)

    corr_summary_df = (
        corr_per_anchor_df
        .groupby("bin_index")
        .agg(
            mean_corr=("correlation", "mean"),
            std_corr=("correlation", "std"),
            valid_anchors=("correlation", lambda x: int(x.notna().sum())),
            mean_tree_range=("tree_range", "mean"),
            mean_hyp_range=("hyp_range", "mean"),
        )
        .reset_index()
    )

    global_rho = spearmanr(
        points_df["tree_distance"],
        points_df["hyperbolic_distance"]
    ).correlation

    global_r = pearsonr(
        points_df["tree_distance"],
        points_df["hyperbolic_distance"]
    )[0]

    # ========================================================
    # Plot
    # ========================================================
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=[
            f"Tree vs hyperbolic distance<br>Spearman={global_rho:.3f}, Pearson={global_r:.3f}, n={len(points_df)}",
            f"Within-bin {correlation_type} correlation<br>Mean over anchors ± std",
        ],
        horizontal_spacing=0.18,
    )

    # Left scatter
    fig.add_trace(
        go.Scattergl(
            x=points_df["tree_distance"],
            y=points_df["hyperbolic_distance"],
            mode="markers",
            marker=dict(
                size=marker_size,
                opacity=opacity,
                color=points_df["bin_index"],
                colorscale="Viridis",
                colorbar=dict(title="Distance bin", x=0.46),
            ),
            text=[
                f"anchor={a}<br>node={n}<br>bin={b}<br>tree={td:.4f}<br>hyp={hd:.4f}"
                for a, n, b, td, hd in zip(
                    points_df["anchor"],
                    points_df["node"],
                    points_df["bin_index"],
                    points_df["tree_distance"],
                    points_df["hyperbolic_distance"],
                )
            ],
            hoverinfo="text",
            name="sampled pairs",
        ),
        row=1,
        col=1,
    )

    # Right line + std band
    x = corr_summary_df["bin_index"].to_numpy()
    mean = corr_summary_df["mean_corr"].to_numpy()
    std = corr_summary_df["std_corr"].fillna(0.0).to_numpy()

    upper = mean + std
    lower = mean - std

    fig.add_trace(
        go.Scatter(
            x=x,
            y=upper,
            mode="lines",
            line=dict(width=0),
            showlegend=False,
            hoverinfo="skip",
        ),
        row=1,
        col=2,
    )

    fig.add_trace(
        go.Scatter(
            x=x,
            y=lower,
            mode="lines",
            fill="tonexty",
            line=dict(width=0),
            name="±1 std",
            opacity=0.25,
            hoverinfo="skip",
        ),
        row=1,
        col=2,
    )

    fig.add_trace(
        go.Scatter(
            x=x,
            y=mean,
            mode="lines+markers",
            marker=dict(size=5),
            line=dict(width=2),
            name=f"mean {correlation_type}",
            text=[
                f"bin={b}<br>mean={m:.3f}<br>std={s:.3f}<br>valid anchors={va}<br>mean tree range={tr:.4f}"
                for b, m, s, va, tr in zip(
                    corr_summary_df["bin_index"],
                    corr_summary_df["mean_corr"],
                    corr_summary_df["std_corr"].fillna(0.0),
                    corr_summary_df["valid_anchors"],
                    corr_summary_df["mean_tree_range"],
                )
            ],
            hoverinfo="text",
        ),
        row=1,
        col=2,
    )

    fig.update_xaxes(title_text="Tree distance", row=1, col=1)
    fig.update_yaxes(title_text="Hyperbolic distance", row=1, col=1)

    fig.update_xaxes(title_text="Distance bin", row=1, col=2)
    fig.update_yaxes(title_text=f"{correlation_type.capitalize()} correlation", range=[0, 1], row=1, col=2)

    fig.update_layout(
        width=1400,
        height=650,
        template="plotly_white",
        title="Tree-distance calibration and within-bin consistency",
    )

    fig.show()

    return points_df, corr_summary_df, corr_per_anchor_df, fig

In [ ]:
points_df, corr_summary_df, corr_per_anchor_df, fig = (
    plot_all_bins_scatter_with_bin_correlations_hyperbolic_plotly(
        model=model,
        D=D,
        sorted_ids=sorted_ids,
        bin_offsets=bin_offsets,
        node_ids=leaf_ids,
        num_anchors=100,
        samples_per_bin_per_anchor=100,
        device=DEVICE,
        eps=BALL_EPS,
        correlation_type="spearman",
    )
)

In [ ]:
# ============================================================
# 16. Save model
# ============================================================
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = output_dir / CHECKPOINT_NAME

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "num_nodes": num_nodes,
        "emb_dim": EMB_DIM,
        "tree_distance_scale": TREE_DISTANCE_SCALE,
        "ball_eps": BALL_EPS,
    },
    checkpoint_path,
)

print("Saved hyperbolic model.")